# Análise NLP dos Planos de Governo — Eleições 2026

Este notebook apresenta a análise textual dos planos de governo das candidaturas à Presidência da República com **registro deferido** no recorte de dados utilizado.

Os documentos foram obtidos a partir dos dados oficiais disponibilizados pelo Tribunal Superior Eleitoral (TSE). A implementação do pipeline foi mantida em `src/`; este notebook concentra a **análise, os resultados e a interpretação metodológica**.

## Objetivos

- caracterizar o corpus textual;
- identificar termos lexicalmente distintivos com TF-IDF;
- analisar proximidade lexical entre os documentos;
- documentar a busca semântica baseada em embeddings;
- comparar duas configurações de chunking;
- avaliar, manualmente, o comportamento do pipeline RAG;
- registrar limitações e critérios de interpretação.

> As análises são descritivas. Similaridade lexical ou vetorial não representa concordância política, equivalência de propostas, qualidade do plano ou recomendação de candidatura.

## 1. Configuração e carregamento

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path("..")
GOVERNMENT_PLANS_DIR = PROJECT_ROOT / "data" / "processed" / "government_plans"
CORPUS_PATH = GOVERNMENT_PLANS_DIR / "government_plans_corpus.csv"
TEXT_DIR = GOVERNMENT_PLANS_DIR / "text"
EMBEDDINGS_DIR = GOVERNMENT_PLANS_DIR / "embeddings"

pd.set_option("display.max_colwidth", 120)

In [ ]:
corpus = pd.read_csv(CORPUS_PATH)

required_columns = {
    "SQ_CANDIDATO",
    "NM_URNA_CANDIDATO",
    "SG_PARTIDO",
    "ARQUIVO_TEXTO",
    "QT_PAGINAS",
    "QT_PALAVRAS",
    "QT_CARACTERES",
}

missing_columns = required_columns - set(corpus.columns)
assert not missing_columns, f"Colunas ausentes: {sorted(missing_columns)}"
assert corpus["SQ_CANDIDATO"].is_unique
assert len(corpus) == 12

corpus["TEXTO"] = corpus["ARQUIVO_TEXTO"].apply(
    lambda file_name: (TEXT_DIR / file_name).read_text(encoding="utf-8")
)

assert corpus["TEXTO"].str.strip().ne("").all()

print(f"Documentos carregados: {len(corpus)}")
print(f"Total de páginas: {corpus['QT_PAGINAS'].sum():,}")
print(f"Total de palavras: {corpus['QT_PALAVRAS'].sum():,}")

## 2. Caracterização do corpus

Os planos possuem extensões muito diferentes. Essa heterogeneidade é relevante porque documentos maiores geram mais chunks e, em uma busca global, podem aparecer mais vezes entre os resultados recuperados.

In [ ]:
corpus_summary = (
    corpus[
        [
            "NM_URNA_CANDIDATO",
            "SG_PARTIDO",
            "QT_PAGINAS",
            "QT_PALAVRAS",
            "QT_CARACTERES",
        ]
    ]
    .sort_values("QT_PALAVRAS", ascending=False)
    .reset_index(drop=True)
)

corpus_summary

In [ ]:
corpus[["QT_PAGINAS", "QT_PALAVRAS", "QT_CARACTERES"]].describe().round(2)

In [ ]:
plot_data = corpus.sort_values("QT_PALAVRAS")

plt.figure(figsize=(10, 6))
plt.barh(plot_data["NM_URNA_CANDIDATO"], plot_data["QT_PALAVRAS"])
plt.xlabel("Quantidade de palavras")
plt.ylabel("Candidato")
plt.title("Extensão dos planos de governo")
plt.tight_layout()
plt.show()

## 3. Análise lexical com TF-IDF

Para a análise lexical, o texto é normalizado e representado por TF-IDF. Stopwords em português, termos editoriais e identificadores dos próprios candidatos/partidos são removidos para reduzir elementos com baixo valor discriminativo.

O TF-IDF mede **importância relativa de termos no corpus**. Portanto, os termos destacados não devem ser interpretados como as propostas mais importantes de uma candidatura.

In [ ]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^\w\sáàâãéèêíïóôõöúç]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

corpus["TEXTO_NORMALIZADO"] = corpus["TEXTO"].apply(normalize_text)

In [ ]:
import nltk
from nltk.corpus import stopwords

try:
    portuguese_stopwords = set(stopwords.words("portuguese"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    portuguese_stopwords = set(stopwords.words("portuguese"))

custom_stopwords = {
    "governo", "plano", "programa", "brasil", "brasileiro", "brasileira",
    "brasileiros", "brasileiras", "livro", "capítulo", "página", "páginas",
    "índice", "sumário",
}

candidate_stopwords = set()
for candidate_name in corpus["NM_URNA_CANDIDATO"]:
    candidate_stopwords.update(normalize_text(candidate_name).split())

party_stopwords = set(corpus["SG_PARTIDO"].str.lower())
final_stopwords = portuguese_stopwords | custom_stopwords | candidate_stopwords | party_stopwords

vectorizer = TfidfVectorizer(
    stop_words=list(final_stopwords),
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 2),
)

tfidf_matrix = vectorizer.fit_transform(corpus["TEXTO_NORMALIZADO"])
feature_names = vectorizer.get_feature_names_out()

print("Dimensão da matriz TF-IDF:", tfidf_matrix.shape)

In [ ]:
def get_top_terms(row_index: int, top_n: int = 10) -> list[tuple[str, float]]:
    scores = tfidf_matrix[row_index].toarray().ravel()
    top_indices = scores.argsort()[-top_n:][::-1]
    return [(feature_names[index], scores[index]) for index in top_indices]

records = []
for index, row in corpus.iterrows():
    for rank, (term, score) in enumerate(get_top_terms(index), start=1):
        records.append(
            {
                "CANDIDATO": row["NM_URNA_CANDIDATO"],
                "PARTIDO": row["SG_PARTIDO"],
                "RANK": rank,
                "TERMO": term,
                "TF_IDF": score,
            }
        )

top_terms = pd.DataFrame(records)
top_terms

## 4. Similaridade lexical

A similaridade cosseno abaixo é calculada sobre a representação TF-IDF. Valores maiores indicam maior proximidade na distribuição dos termos utilizados.

**Importante:** proximidade lexical não implica equivalência de propostas, alinhamento ideológico ou concordância política.

In [ ]:
similarity_matrix = cosine_similarity(tfidf_matrix)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=corpus["NM_URNA_CANDIDATO"],
    columns=corpus["NM_URNA_CANDIDATO"],
)

upper_triangle = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]
print(pd.Series(upper_triangle).describe().round(3))

fig, ax = plt.subplots(figsize=(11, 9))
image = ax.imshow(similarity_matrix)
ax.set_xticks(range(len(corpus)))
ax.set_yticks(range(len(corpus)))
ax.set_xticklabels(corpus["NM_URNA_CANDIDATO"], rotation=90)
ax.set_yticklabels(corpus["NM_URNA_CANDIDATO"])
fig.colorbar(image, ax=ax, label="Similaridade cosseno")
ax.set_title("Similaridade lexical entre planos de governo")
plt.tight_layout()
plt.show()

In [ ]:
pairs = []
for i in range(len(corpus)):
    for j in range(i + 1, len(corpus)):
        pairs.append(
            {
                "DOCUMENTO_A": corpus.iloc[i]["NM_URNA_CANDIDATO"],
                "DOCUMENTO_B": corpus.iloc[j]["NM_URNA_CANDIDATO"],
                "SIMILARIDADE": similarity_matrix[i, j],
            }
        )

similarity_pairs = (
    pd.DataFrame(pairs)
    .sort_values("SIMILARIDADE", ascending=False)
    .reset_index(drop=True)
)

similarity_pairs.head(10).round(3)

No experimento realizado durante o desenvolvimento, as 66 combinações de documentos apresentaram similaridade lexical média próxima de **0,245**, com valores entre aproximadamente **0,003 e 0,540**. Esses valores descrevem somente a representação lexical utilizada.

## 5. Busca semântica com embeddings

Para recuperar trechos semanticamente relacionados a uma consulta, o pipeline final utiliza:

```text
PDFs oficiais
    ↓
pypdf
    ↓
Corpus textual
    ↓
Chunking: 500 palavras / overlap 75
    ↓
qwen3-embedding:0.6b
    ↓
Similaridade cosseno
    ↓
Top-k = 3
    ↓
Gemma 3 4B
    ↓
Resposta baseada nas fontes recuperadas
```

A configuração final de chunking foi escolhida após uma comparação controlada com o baseline de 1.000 palavras e overlap 150. Os embeddings e os metadados completos dos chunks são artefatos reprodutíveis e, por isso, não precisam ser regenerados neste notebook.

In [ ]:
config_path = EMBEDDINGS_DIR / "embedding_config.json"

if config_path.exists():
    import json
    embedding_config = json.loads(config_path.read_text(encoding="utf-8"))
    display(pd.Series(embedding_config, name="valor").to_frame())
else:
    print("embedding_config.json não encontrado. Execute o pipeline de embeddings em src/analysis/.")

## 6. Avaliação do retrieval

A recuperação foi avaliada manualmente em **8 consultas temáticas**, com **5 resultados por consulta** (40 resultados por configuração). Um resultado foi marcado como relevante quando o chunk continha conteúdo substantivo diretamente relacionado ao tema consultado.

Essa avaliação mede **Precision@5 no conjunto analisado**. Ela não representa “acurácia dos embeddings” nem prova desempenho perfeito em consultas não avaliadas.

In [ ]:
retrieval_comparison = pd.DataFrame(
    {
        "Configuração": ["1000 / 150", "500 / 75"],
        "Chunks": [302, 598],
        "Consultas": [8, 8],
        "Resultados avaliados": [40, 40],
        "Resultados relevantes": [39, 40],
        "Precision@5 média": [0.975, 1.000],
    }
)

retrieval_comparison

In [ ]:
retrieval_final_path = EMBEDDINGS_DIR / "retrieval_evaluation_500_75.csv"

if retrieval_final_path.exists():
    retrieval_final = pd.read_csv(retrieval_final_path)
    display(
        retrieval_final[
            [
                "QUERY_ID", "RANK", "NM_URNA_CANDIDATO", "SG_PARTIDO",
                "CHUNK_ID", "SIMILARIDADE", "RELEVANTE"
            ]
        ].head(15)
    )
else:
    print("Arquivo de avaliação do retrieval não encontrado.")

A configuração `500/75` recuperou 40 de 40 chunks considerados relevantes no conjunto manual. Entretanto, ainda foram observados dois pontos importantes:

- chunks de janela fixa podem atravessar fronteiras temáticas;
- documentos maiores podem gerar mais chunks e concentrar resultados de uma busca global.

Por isso, `Precision@5 = 1,000` deve ser lido estritamente no contexto das 8 consultas avaliadas.

## 7. Avaliação do RAG

O gerador foi avaliado separadamente do retrieval. Foram utilizados quatro critérios manuais:

- **Grounding:** a resposta permanece sustentada pelas fontes recuperadas;
- **Citação:** utiliza corretamente marcadores `[FONTE X]`;
- **Português:** responde em português;
- **Sem avaliação:** não produz ranking, recomendação ou julgamento das candidaturas.

As taxas abaixo são **taxas de conformidade por critério em 8 consultas**, e não uma medida geral de acurácia.

In [ ]:
rag_comparison = pd.DataFrame(
    {
        "Modelo": ["Llama 3.2 3B", "Gemma 3 4B"],
        "Grounding": [0.500, 0.875],
        "Citação [FONTE X]": [0.000, 1.000],
        "Português": [1.000, 1.000],
        "Sem avaliação/ranking": [1.000, 1.000],
    }
)

rag_comparison.style.format(
    {
        "Grounding": "{:.1%}",
        "Citação [FONTE X]": "{:.1%}",
        "Português": "{:.1%}",
        "Sem avaliação/ranking": "{:.1%}",
    }
)

In [ ]:
gemma_manual_evaluation = pd.DataFrame(
    [
        ("educacao", 1, 1, 1, 1, "Propostas separadas e citadas."),
        ("saude", 1, 1, 1, 1, "Conteúdo sustentado; resposta mais longa."),
        ("seguranca", 1, 1, 1, 1, "Grounded; inclui conteúdo periférico de um chunk."),
        ("trabalho", 1, 1, 1, 1, "Síntese sustentada pelas fontes."),
        ("meio_ambiente", 1, 1, 1, 1, "Conteúdo e atribuições preservados."),
        ("moradia", 0, 1, 1, 1, "Inversão de sentido em uma passagem recuperada."),
        ("infraestrutura", 1, 1, 1, 1, "Conteúdo sustentado pelas fontes."),
        ("mulheres", 1, 1, 1, 1, "Conteúdo sustentado pelas fontes."),
    ],
    columns=[
        "CONSULTA", "GROUNDED", "CITACAO", "PORTUGUES",
        "SEM_AVALIACAO", "OBSERVACAO"
    ],
)

gemma_manual_evaluation

### Falha de grounding observada

A consulta sobre moradia mostrou por que citação correta não é suficiente para garantir fidelidade semântica. Em uma das fontes, a frase criticava a decisão de fila habitacional “no balcão” como espaço para clientelismo. A resposta gerada transformou essa crítica em uma proposta, invertendo o sentido do trecho.

Esse caso foi marcado como `GROUNDED = 0`, mesmo com a presença correta de `[FONTE X]`. O exemplo evidencia a necessidade de avaliar **atribuição e sentido**, não apenas fluência e citação.

## 8. Configuração final do experimento

Após os testes controlados, a configuração mantida para a POC é:

| Componente | Configuração |
|---|---|
| Extração de texto | `pypdf` |
| Chunking | 500 palavras |
| Overlap | 75 palavras |
| Embeddings | `qwen3-embedding:0.6b` |
| Recuperação | similaridade cosseno |
| Contexto do RAG | top-3 chunks |
| Gerador | `gemma3:4b` |

O Llama 3.2 3B permanece documentado como experimento comparativo, não como gerador final.

## 9. Limitações

Os resultados devem ser interpretados considerando as seguintes limitações:

1. **Tamanho do conjunto de avaliação:** retrieval e RAG foram avaliados em apenas 8 consultas temáticas.
2. **Rotulagem manual:** a relevância e o grounding dependem de critérios definidos para esta POC.
3. **Chunking por palavras:** janelas fixas podem misturar assuntos quando uma fronteira temática ocorre dentro do chunk.
4. **Comprimento desigual dos documentos:** planos maiores produzem mais chunks e podem ter maior presença no top-k global.
5. **PDFs:** a extração textual pode preservar ruídos, duplicações e estruturas editoriais do documento original.
6. **TF-IDF:** similaridade lexical não equivale a similaridade ideológica ou equivalência de propostas.
7. **Embeddings:** proximidade vetorial indica proximidade na representação do modelo, não qualidade ou concordância política.
8. **RAG:** respostas fluentes e corretamente citadas ainda podem distorcer o sentido de uma fonte.

A POC demonstra um pipeline reprodutível para exploração documental, mas não substitui leitura integral dos planos nem análise especializada.

## 10. Conclusão

A análise combina duas perspectivas complementares:

- **NLP lexical**, com TF-IDF e similaridade cosseno, para observar padrões de vocabulário;
- **NLP semântico**, com embeddings e RAG, para recuperar e sintetizar trechos relacionados a consultas temáticas.

No conjunto manual de retrieval, a mudança de `1000/150` para `500/75` elevou a Precision@5 observada de **0,975 para 1,000**. Na avaliação manual do gerador, o Gemma 3 4B apresentou **87,5% de conformidade de grounding** e **100% de conformidade de citação** nas 8 consultas, mas ainda apresentou uma inversão de sentido relevante.

O principal resultado metodológico é que **retrieval e geração precisam ser avaliados separadamente**: recuperar trechos relevantes não garante que o modelo gerador preserve corretamente seu significado.